In [26]:
import pandas as pd
from dateutil.parser import parse
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime

In [18]:
# Cargar dataset
dataset_path = 'customer_satisfaction_data.csv'
df = pd.read_csv(dataset_path)

In [19]:
# Mostrar primeras filas del dataset limpio
print(df.head())

  CustomerID  Age  PlanType  MonthlyBill  HoursWatched_Last30d SignupDate  \
0     28-AHF   77  estandar        38.37                  36.1  7/10/2023   
1    107-FPV   33    Basico        16.97                  23.6   1/6/2025   
2     11-SNZ   50    Basico        18.28                  34.7  1/21/2021   
3     25-JLV   53    Basico        16.69                  38.1  1/21/2022   
4     56-RVU   28    Basico          NaN                  22.6  5/21/2024   

  CustomerCode                                  LastSupportTicket  
0       28-AHF  ERROR EN LA APLICACION MOVIL. El cliente repor...  
1      107-FPV  CONSULTA SOBRE CONTENIDO DISPONIBLE. El proble...  
2       11-SNZ  problema de conexion a internet. El problema p...  
3       25-JLV  Queja sobre atencion al cliente. Cliente insat...  
4       56-RVU                    Queja sobre atencion al cliente  


In [20]:
# 1. Unificación y limpieza de PlanType
# Poner en minúsculas y eliminar espacios para unificar categorías
df['PlanType_clean'] = df['PlanType'].str.lower().str.strip()

# Corrección manual de categorías erróneas o abreviaturas
df['PlanType_clean'] = df['PlanType_clean'].replace({
    'prem': 'premium',
    'premi': 'premium'
})


In [21]:
# 2. Conversión de SignupDate a formato datetime para manipulación
def try_parse_date(date_str):
    try:
        return parse(date_str, dayfirst=False)
    except:
        return pd.NaT

df['SignupDate_clean'] = df['SignupDate'].apply(try_parse_date)

In [22]:
# 3. Truncamiento de outliers en Age
# Limitar valores mínimos a 10 y máximos a 100
df['Age_clean'] = df['Age'].clip(lower=10, upper=100)

In [23]:
# 4. Imputación de valores nulos en MonthlyBill con mediana
median_monthly_bill = df['MonthlyBill'].median()
df['MonthlyBill_clean'] = df['MonthlyBill'].fillna(median_monthly_bill)

In [24]:
# 5. Codificación de PlanType con etiquetas numéricas (label encoding)
df['PlanType_label'] = df['PlanType_clean'].astype('category').cat.codes

In [27]:
# 6. Calcular la antigüedad del cliente en meses
df['Antiguedad_Cliente'] = ((datetime.now() - df['SignupDate_clean']).dt.days / 30.44).astype(int)

In [28]:
# Dataset limpio listo para usar en el entrenamiento
cols_finales = ['CustomerID', 'Age_clean', 'PlanType_label', 'MonthlyBill_clean',
                'HoursWatched_Last30d', 'SignupDate_clean', 'LastSupportTicket', 'Antiguedad_Cliente']
df_final = df[cols_finales]

In [29]:
# Mostrar primeras y ultimas filas del dataset limpio
print(df_final.head())
print(df_final.tail())

  CustomerID  Age_clean  PlanType_label  MonthlyBill_clean  \
0     28-AHF         77               1              38.37   
1    107-FPV         33               0              16.97   
2     11-SNZ         50               0              18.28   
3     25-JLV         53               0              16.69   
4     56-RVU         28               0              25.59   

   HoursWatched_Last30d SignupDate_clean  \
0                  36.1       2023-07-10   
1                  23.6       2025-01-06   
2                  34.7       2021-01-21   
3                  38.1       2022-01-21   
4                  22.6       2024-05-21   

                                   LastSupportTicket  Antiguedad_Cliente  
0  ERROR EN LA APLICACION MOVIL. El cliente repor...                  28  
1  CONSULTA SOBRE CONTENIDO DISPONIBLE. El proble...                  10  
2  problema de conexion a internet. El problema p...                  58  
3  Queja sobre atencion al cliente. Cliente insat...          